In [47]:
import pandas as pd
import numpy as np
df1 = pd.read_csv("dataset/weather.csv")
from sklearn.preprocessing import LabelEncoder
df1['Outlook'] = LabelEncoder().fit_transform(df1['Outlook'])
df1['Temp'] = LabelEncoder().fit_transform(df1['Temp'])
df1['Windy'] = LabelEncoder().fit_transform(df1['Windy'])
df1['Humidity'] = LabelEncoder().fit_transform(df1['Humidity'])
df1['Play'] = LabelEncoder().fit_transform(df1['Play'])
df1
df = pd.read_csv("dataset/weather.csv")
df

,Outlook,Temp,Humidity,Windy,Play
0,sunny,hot,high,weak,no
1,sunny,hot,high,strong,no
2,overcast,hot,high,weak,yes
3,rainy,mild,high,weak,yes
4,rainy,cool,normal,weak,yes
5,rainy,cool,normal,strong,no
6,overcast,cool,normal,strong,yes
7,sunny,mild,high,weak,no
8,sunny,cool,normal,weak,yes
9,rainy,mild,normal,weak,yes


In [25]:
def find_entropy(df):
    target = df.keys()[-1]
    targ_values = df[target].unique()
    entropy = 0
    for targ_value in targ_values:
        prob = len(df[df[target] == targ_value])/len(df)
        entropy+= prob*np.log2(prob)
    return entropy

In [26]:
find_entropy(df1)

np.float64(-0.9402859586706311)

In [27]:
def find_average_info_entropy(df,attribute):
    attr_values = df[attribute].unique()
    target = df.keys()[-1]
    targ_values = df[target].unique()
    avg_info_entropy = 0
    for attr_value in attr_values:
        entropy_subsample = 0
        for targ_value in targ_values:
            num = len(df[(df[attribute] == attr_value) & (df[target] == targ_value)])
            den = len(df[df[attribute] == attr_value])
            prob = num / den
            entropy_subsample+=-prob * np.log2(prob+1e-7)
        avg_info_entropy+= (len(df[df[attribute] == attr_value])/len(df))*entropy_subsample
    return avg_info_entropy

In [28]:
find_average_info_entropy(df,'Outlook')

np.float64(0.6935358915770655)

In [29]:
def IG_selection(df,k):
    cols = df.keys()
    attributes = cols[:-1]
    target = cols[-1]
    IG = {}
    for attribute in attributes:
        IG[attribute] = find_entropy(df) - find_average_info_entropy(df, attribute)
    sorted_values = sorted(IG.items(),key=lambda x : x[1],reverse=True)
    selected_features = [f for f,_ in sorted_values[:k]]
    return selected_features

In [30]:
IG_selection(df,2)

['Outlook', 'Humidity']

In [31]:
def find_winner(df):
    attributes =df.keys()[:-1]
    IG = []
    for attribute in attributes:
        IG.append(find_entropy(df)-find_average_info_entropy(df,attribute))
    return attributes[np.argmax(IG)]

In [57]:
def find_winner_rf(df,feature_subset):
    # attributes =df.keys()[:-1]
    IG = []
    for attribute in feature_subset:
        IG.append(find_entropy(df)-find_average_info_entropy(df,attribute))
    return feature_subset[np.argmax(IG)]

In [59]:
find_winner_rf(df,['Outlook','Temp'])

'Outlook'

In [32]:
find_winner(df)

'Outlook'

In [82]:
def training_rf(df,feature_subset):
    target = df.keys()[-1]
    if len(df[target].unique()) == 1:
        return df[target].unique()[0]
    if len(df.columns) == 1:
        return df[target].mode()[0]
    best_feature = find_winner_rf(df,feature_subset)
    tree = {best_feature:{}}
    for value in df[best_feature].unique():
        sub_df = df[df[best_feature] == value].drop(columns=[best_feature])
        tree[best_feature][value]    = training_rf(sub_df,feature_subset)
    return tree

In [94]:
def training(df):
    target = df.keys()[-1]
    attribus = df.keys()[:-1]
    if len(df[target].unique()) == 1:
        return df[target].unique()[0]
    if len(df.columns) == 1 or len(feature_subset) == 0:
        return df[target].mode()[0]
    best_feature = find_winner_rf(df)
    tree = {best_feature:{}}
    for value in df[best_feature].unique():
        sub_df = df[df[best_feature] == value].drop(columns=[best_feature])
        feature_subset = [f for f in feature_subset if f != best_feature]
        tree[best_feature][value]    = training_rf(sub_df,feature_subset)
    return tree

In [95]:
tree = training(df)

UnboundLocalError: cannot access local variable 'feature_subset' where it is not associated with a value

In [96]:
def predict(tree,instance):
    for attribute in tree.keys():
        value = instance[attribute]
        sub_tree = tree[attribute][value]
        if type(sub_tree) is dict:
            return predict(sub_tree,instance)
        else:
            return sub_tree
    return 0

In [97]:
instances = [
    {'Outlook': 'sunny', 'Humidity': 'high', 'Wind': 'weak'} # unseen value
]

for inst in instances:
    print(inst, '→', predict(tree, inst))

{'Outlook': 'sunny', 'Humidity': 'high', 'Wind': 'weak'} → no


In [98]:
def bootstrap(df):
    return df.sample(int(0.8*len(df)),replace=True)

In [99]:
bootstrap(df)

,outlook,temp,humidity,windy,play
6,overcast,cool,normal,strong,yes
8,sunny,cool,normal,weak,yes
1,sunny,hot,high,strong,no
0,sunny,hot,high,weak,no
7,sunny,mild,high,weak,no
7,sunny,mild,high,weak,no
13,rainy,mild,high,strong,no
9,rainy,mild,normal,weak,yes
7,sunny,mild,high,weak,no
3,rainy,mild,high,weak,yes


In [100]:
def training_rf(df,feature_subset):
    target = df.keys()[-1]
    if len(df[target].unique()) == 1:
        return df[target].unique()[0]
    if len(df.columns) == 1:
        return df[target].mode()[0]
    best_feature = find_winner_rf(df,feature_subset)
    tree = {best_feature:{}}
    for value in df[best_feature].unique():
        sub_df = df[df[best_feature] == value].drop(columns=[best_feature])
        tree[best_feature][value]    = training_rf(sub_df,feature_subset)
    return tree

In [101]:
import random
def build_random_forest(df,n_trees,max_features=None):
    trees = []
    attributes = list(df.keys()[:-1])
    for i in range(n_trees):
        subsample = bootstrap(df)
        if max_features is None:
            max_features = int(np.sqrt(len(attributes)))
        feature_subset = random.sample(attributes,max_features)
        trees.append(training_rf(subsample,feature_subset))
    return trees

In [102]:
df.columns = df.columns.str.lower()
build_random_forest(df,1)

KeyError: 'windy'

In [93]:
for i in range(len(df1)):
    predicted_labels = [predict(tree,df1.iloc[i]) for tree in trees]
    Y_label = Counter(predicted_labels).most_common()[0][0]

NameError: name 'trees' is not defined

In [91]:
from collections import  Counter
predicted_labels=['yes','no','no']
Counter(predicted_labels).most_common()[0][0]

'no'